# backprop-pop-outgrad-loop — ex3: instrument reverse-pass with per-node max|grad_out| trace

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backprop-pop-outgrad-loop`. Running the final beacon cell reports progress against the `Backprop: backprop pop-outgrad loop` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backprop pop-outgrad loop` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backprop-pop-outgrad-loop`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backprop-pop-outgrad-loop"
DD_SUBTOPIC = "Backprop: backprop pop-outgrad loop"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Per-node max|grad| trace — vanishing/exploding diagnostic

Ex1 ran the reverse-pass driver; ex2 counted per-leaf accumulations.
The deepening move tracks the L∞ MAGNITUDE of each `grad_out` as it
flows back — a fingerprint of vanishing-gradient (values → 0 deep in
the graph) or exploding-gradient (values → ∞) pathologies.

```python
def backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    trace = []  # [(node_id, max_abs_grad_out), ...] in pop order
    for node in sorted_graph:
        if id(node) not in grads: continue
        grad_out = grads.pop(id(node))
        trace.append((id(node), float(grad_out.abs().max())))
        # ... rest of driver as ex1 ...
    return trace
```

**Why max|.| not norm.** L∞ catches a SINGLE explosive element — one
rogue activation that's about to overflow on the next forward pass.
L2 averages it out. For health monitoring, the worst element is the
right signal.

**Trace order matches reverse-pass order.** The first entry is
`end_node`; the last entries are leaves. A monotone-decreasing trace
is the vanishing signature; an increasing one is exploding. Most real
graphs show a noisy mix — but a clean monotone pattern over 50+ layers
is what RNN tutorials famously visualize.

### Exercise 3 — instrument reverse-pass with per-node max|grad_out| trace

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the reverse pass by recording the L∞ magnitude (`grad_out.abs().max().item()`) for every node as it's popped, producing a trace list that fingerprints vanishing/exploding gradient pathologies.
> Keywords: trace, max-abs, vanishing-gradient, exploding-gradient, diagnostic
> ```

**KCs targeted:** `backprop-pop-outgrad-loop`, `linf-grad-trace-per-node`

Implement `ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs)` — the same reverse-pass driver as ex1, but also return a `trace` list capturing the L∞ magnitude of each node's popped `grad_out`.

Signature:

```python
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    ...
    return trace  # list[tuple[int, float]]
```

Semantics:
- `trace` is a list of `(id(node), float(grad_out.abs().max()))` entries, appended IMMEDIATELY AFTER popping `grad_out` from the accumulator and BEFORE dispatching to back_fns.
- Only nodes that ACTUALLY get popped (i.e. have a grad routed to them) appear in the trace. Skipped nodes (no entry in `grads`) are absent.
- Order: matches the reverse-pass walk through `sorted_graph` — `end_node` typically first, leaves last.

Same three reverse-pass invariants as ex1:
1. POP (don't peek) the entry in `grads`.
2. ACCUMULATE (don't overwrite) — `+=` per parent.
3. LEAVES write `.grad`; non-leaves stay in the `grads` dict.

Mutate `.grad` on leaves in place; return only `trace`.

In [ ]:
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs) -> list:
    """Reverse-pass driver + per-node max|grad_out| trace."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        from types import SimpleNamespace

        # === MiniTensor scaffold ===
        def leaf(value):
            return SimpleNamespace(array=t.tensor(value), recipe=None, grad=None)
        def interior(value, func, parents, args=(), kwargs=None):
            return SimpleNamespace(
                array=t.tensor(value),
                recipe=SimpleNamespace(func=func, parents=parents, args=args, kwargs=kwargs or {}),
                grad=None,
            )

        # === Chain: x -> y = x*2 -> z = y*3.  d(z)/d(x) = 6 ===
        x = leaf([1.0, 2.0, 3.0])
        y = interior([2.0, 4.0, 6.0], func='mul', parents={0: x}, args=(x.array, 2.0))
        z = interior([6.0, 12.0, 18.0], func='mul', parents={0: y}, args=(y.array, 3.0))

        def mul_back(grad_out, out_, x_, c):
            return grad_out * c

        back_funcs = {('mul', 0): mul_back}
        sorted_graph = [z, y, x]   # reverse-topological
        end_grad = t.ones(3)

        trace = ex3_backprop_traced(z, end_grad, sorted_graph, back_funcs)

        # === Trace shape: list of (id, float) ===
        assert isinstance(trace, list), f'trace must be list; got {type(trace).__name__}'
        assert all(isinstance(e, tuple) and len(e) == 2 for e in trace), trace
        assert all(isinstance(e[0], int) and isinstance(e[1], float) for e in trace), trace

        # === Three entries, one per popped node ===
        assert len(trace) == 3, f'expected 3 trace entries, got {len(trace)}: {trace}'

        # === Order matches reverse walk ===
        ids = [e[0] for e in trace]
        assert ids == [id(z), id(y), id(x)], f'trace order wrong: {ids}'

        # === Magnitudes: z gets end_grad (max=1), y gets 3*end_grad (max=3), x gets 6 (max=6) ===
        mags = [e[1] for e in trace]
        assert abs(mags[0] - 1.0) < 1e-6, f'z mag should be 1.0; got {mags[0]}'
        assert abs(mags[1] - 3.0) < 1e-6, f'y mag should be 3.0; got {mags[1]}'
        assert abs(mags[2] - 6.0) < 1e-6, f'x mag should be 6.0; got {mags[2]}'

        # === Leaf .grad written correctly ===
        assert x.grad is not None, 'leaf .grad must be populated'
        assert t.allclose(x.grad, t.tensor([6.0, 6.0, 6.0])), x.grad

        # === Diamond DAG: out = x*x, accumulation via both parents ===
        x = leaf([2.0])
        sq = interior([4.0], func='mul', parents={0: x, 1: x}, args=(x.array, x.array))
        def mul_back_0(grad_out, out_, a, b):
            return grad_out * b
        def mul_back_1(grad_out, out_, a, b):
            return grad_out * a
        back_funcs = {('mul', 0): mul_back_0, ('mul', 1): mul_back_1}
        trace = ex3_backprop_traced(sq, t.ones(1), [sq, x], back_funcs)
        assert len(trace) == 2, f'expected 2 trace entries; got {len(trace)}'
        assert trace[0][0] == id(sq) and trace[1][0] == id(x)
        # x's accumulated grad = 2 + 2 = 4 → its popped max is 4.0
        assert abs(trace[1][1] - 4.0) < 1e-6, f'x mag should be 4.0; got {trace[1][1]}'
        assert t.allclose(x.grad, t.tensor([4.0])), x.grad

        # === Decreasing trace = vanishing signature ===
        # Chain of x4 — start with grad 1.0, accumulate scaling of 0.5 each step
        a = leaf([1.0])
        b = interior([0.5], func='scale', parents={0: a}, args=(a.array,))
        c = interior([0.25], func='scale', parents={0: b}, args=(b.array,))
        d = interior([0.125], func='scale', parents={0: c}, args=(c.array,))
        def scale_back(grad_out, out_, x_):
            return grad_out * 0.5
        back_funcs = {('scale', 0): scale_back}
        trace = ex3_backprop_traced(d, t.ones(1), [d, c, b, a], back_funcs)
        # mags should be 1.0, 0.5, 0.25, 0.125 — monotone-decreasing = vanishing.
        expected = [1.0, 0.5, 0.25, 0.125]
        for i, (got_id, got_mag) in enumerate(trace):
            assert abs(got_mag - expected[i]) < 1e-6, f'mag {i}: expected {expected[i]}, got {got_mag}'
        # And it's strictly monotone — the vanishing fingerprint.
        mags = [e[1] for e in trace]
        assert all(mags[i] > mags[i+1] for i in range(len(mags)-1)), f'vanishing: not monotone: {mags}'
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_backprop_traced(end_node, end_grad, sorted_graph, back_funcs):
    grads = {id(end_node): end_grad}
    trace = []
    for node in sorted_graph:
        if id(node) not in grads:
            continue
        grad_out = grads.pop(id(node))
        # Record L\u221E magnitude AFTER pop, BEFORE dispatch — this is what
        # was routed to this node, including any diamond-DAG accumulation.
        trace.append((id(node), float(grad_out.abs().max().item())))
        if node.recipe is None:
            node.grad = grad_out if node.grad is None else node.grad + grad_out
            continue
        for argnum, parent in node.recipe.parents.items():
            bf = back_funcs[(node.recipe.func, argnum)]
            gp = bf(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
            pid = id(parent)
            grads[pid] = grads.get(pid, 0) + gp
    return trace
```

**Record AFTER pop, BEFORE dispatch.** Recording before pop would miss the accumulation step (diamond DAG: parent gets contributions from multiple children; the magnitude isn't final until the pop). Recording after dispatch would conflate the parent's incoming grad with the post-back_fn grad, which is a DIFFERENT quantity.

**L∞ catches the worst element.** A grad of shape `(1024,)` with 1023 zeros and one `1e10` is exploding in real terms; L2 averages it to `1e10/sqrt(1024) ≈ 3.1e8` which understates the danger. Max-abs is the right alarm signal.

**`.item()` for the float cast.** The trace must be JSON-serializable for downstream logging. Python floats (not 0-D tensors) achieve that.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()